# Experiment: MCMC-IS Beta Diagnostics

Objective:
- Fix one exact scenario.
- Compare MCMC-IS performance across beta values.
- Record estimates and diagnostics at intermediate estimation points.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    MCMCWorkflowConfig,
    SAMCWorkflowConfig,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_selected_scenarios,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
)

pd.set_option("display.max_columns", 100)
project_root

## Configuration

`ESTIMATION_POINTS` controls checkpoint budgets.  
The final checkpoint is used for the main beta boxplots; all checkpoints are used for convergence plots.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_beta_notebook"

SCENARIO_KEY = "rank_sum_dp_n40"
ESTIMATION_POINTS = (10_000, 100_000, 1_000_000) if not FAST_MODE else (2_000, 10_000, 20_000)
BETA_MULTIPLIERS = (0.70, 0.90, 1.00, 1.15, 1.35)
BETA_REPEATS = 5 if not FAST_MODE else 2
BASE_SEED = 54_321

mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=20_000 if not FAST_MODE else 1_000,
    tune_steps=12_000 if not FAST_MODE else 1_000,
    local_scan_total_steps=80_000 if not FAST_MODE else 4_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_fraction=0.075,
)
beta_cfg = BetaSweepStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=BETA_REPEATS,
    beta_multipliers=BETA_MULTIPLIERS,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_fraction=0.075,
    base_seed=BASE_SEED,
)

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_KEY": SCENARIO_KEY,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "BETA_MULTIPLIERS": BETA_MULTIPLIERS,
    "BETA_REPEATS": BETA_REPEATS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
}, indent=2))

## Load Scenario And Build Beta Workflow

In [ ]:
scenario = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=[SCENARIO_KEY],
    min_tail_states=2,
)[0]

beta_workflow = build_beta_workflow(
    scenario.problem,
    scenario.exact_p,
    mcmc_cfg,
    seed=BASE_SEED + 10_000,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "beta_diag") if SAVE_OUTPUTS else None

print(json.dumps({
    "scenario": scenario.key,
    "exact_p": scenario.exact_p,
    "beta0_laplace": beta_workflow["beta0_laplace"],
    "beta_hat_tuned": beta_workflow["beta_hat_tuned"],
    "beta_used": beta_workflow["beta_used"],
    "q_target": beta_workflow["q_target"],
}, indent=2))

## Run Beta Sweep Across Checkpoints

In [ ]:
beta_study = run_beta_checkpoint_study(
    scenario.problem,
    scenario.exact_p,
    beta_center=float(beta_workflow["beta_used"]),
    sigma_t=float(beta_workflow["sigma_t"]),
    beta_cfg=beta_cfg,
)

if SAVE_OUTPUTS and run_dir is not None:
    save_beta_sweep_outputs(
        beta_study,
        output_dir=run_dir / scenario.key,
        scenario_name=scenario.description,
        exact_p=scenario.exact_p,
        beta_cfg=beta_cfg,
        beta_workflow=beta_workflow,
    )

beta_summary_df = pd.DataFrame(beta_study["summary"]).sort_values(["checkpoint", "beta"])
display(beta_summary_df[[
    "checkpoint",
    "beta",
    "mean_estimate",
    "rmse",
    "mean_variance_estimate",
    "mean_q_tilt_tail_share",
    "mean_ess",
    "mean_acceptance_rate",
    "mean_weight_cv",
]])

## Review Saved Figures

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    scenario_dir = run_dir / scenario.key
    print(f"Saved outputs under: {scenario_dir}")
    display(Image(filename=str(scenario_dir / "beta_max_budget.png")))
    display(Image(filename=str(scenario_dir / "beta_convergence.png")))
else:
    print("SAVE_OUTPUTS=False, so no saved figures to display.")

## Reload Saved Results Without Rerunning

In [ ]:
RELOAD_BETA_DIR = None
# Example:
# RELOAD_BETA_DIR = project_root / "results" / "mcmcis_beta_notebook" / "20260306_120000_beta_diag" / "rank_sum_dp_n40"

if RELOAD_BETA_DIR is not None:
    saved = load_beta_sweep_saved_output(RELOAD_BETA_DIR)
    print(json.dumps({
        "scenario_display": saved["metadata"]["scenario_display"],
        "exact_p": saved["metadata"]["exact_p"],
        "beta_center": saved["metadata"]["settings"]["beta_center"],
    }, indent=2))
    regen = regenerate_beta_sweep_plots_from_saved(RELOAD_BETA_DIR)
    for name, path in regen.items():
        print(name, path)
        display(Image(filename=str(path)))
else:
    print("Set RELOAD_BETA_DIR to a saved beta-study directory to regenerate plots from disk only.")